# Preprocessing Pipeline

To prevent data leakage and ensure reproducibility, preprocessing operations are integrated into machine learning pipelines.

The preprocessing framework includes:
- numerical scaling
- categorical encoding
- automated transformation handling
- fold-safe preprocessing during cross-validation

# Imports

In [20]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    RandomizedSearchCV,
    cross_validate
)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    StackingClassifier
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import joblib

In [2]:
train_df = pd.read_csv('/kaggle/input/datasets/reemhatemzekry/titanic-competition-data/processed_train.csv')
test_df = pd.read_csv('/kaggle/input/datasets/reemhatemzekry/titanic-competition-data/processed_test.csv')

In [3]:
test_df.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,...,FamilySize,IsAlone,TicketGroup,Deck,FarePerPerson,AgeBin,Sex_Pclass,Age_Class,Fare_Class,Family_Fare
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,...,1,1,1,U,7.829200,Young Adult,male_3,103.5,23.4876,7.8292
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,...,2,0,1,U,3.500000,Adult,female_3,141.0,21.0000,14.0000
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,...,1,1,1,U,9.687500,Senior,male_2,124.0,19.3750,9.6875
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,...,1,1,1,U,8.662500,Young Adult,male_3,81.0,25.9875,8.6625
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,...,3,0,2,U,4.095833,Young Adult,female_3,66.0,36.8625,36.8625


In [4]:
test_df.isnull().sum()

PassengerId        0
Pclass             0
Name               0
Sex                0
Age                0
SibSp              0
Parch              0
Ticket             0
Fare               0
Cabin            327
Embarked           0
Title              0
FamilySize         0
IsAlone            0
TicketGroup        0
Deck               0
FarePerPerson      0
AgeBin             0
Sex_Pclass         0
Age_Class          0
Fare_Class         0
Family_Fare        0
dtype: int64

In [5]:
print(train_df.shape)
print(test_df.shape)

(891, 23)
(418, 22)


In [6]:
train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,...,FamilySize,IsAlone,TicketGroup,Deck,FarePerPerson,AgeBin,Sex_Pclass,Age_Class,Fare_Class,Family_Fare
0,1,0.0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,...,2,0,1,U,3.62500,Young Adult,male_3,66.0,21.7500,14.5000
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,...,2,0,2,C,35.64165,Adult,female_1,38.0,71.2833,142.5666
2,3,1.0,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,...,1,1,1,U,7.92500,Young Adult,female_3,78.0,23.7750,7.9250
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,...,2,0,2,C,26.55000,Young Adult,female_1,35.0,53.1000,106.2000
4,5,0.0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,...,1,1,1,U,8.05000,Young Adult,male_3,105.0,24.1500,8.0500


# X and Y data split

In [7]:
drop_cols = ["PassengerId", "Name", "Cabin", "Ticket"]

X = train_df.drop(columns=["Survived"] + drop_cols)
y = train_df["Survived"]

X_test = test_df.drop(columns=drop_cols)

passenger_ids = test_df["PassengerId"]

In [8]:
categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()


print("Categorical Features:")
print(categorical_features)

Categorical Features:
['Sex', 'Embarked', 'Title', 'Deck', 'AgeBin', 'Sex_Pclass']


In [9]:
numerical_features = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("\nNumerical Features:")
print(numerical_features)


Numerical Features:
['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'TicketGroup', 'FarePerPerson', 'Age_Class', 'Fare_Class', 'Family_Fare']


# Preprocessing Pipeline

In [10]:
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Cross validation

In [12]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Evaluation metrics

In [18]:
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}


# Baseline Model

## Logistic regression model

In [40]:
from sklearn.model_selection import cross_validate

logistic_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

results = cross_validate(
    logistic_pipeline,
    X,
    y,
    cv=cv,
    scoring=scoring
)

print("Logistic Regression Metrics:")
print(f"Accuracy  : {results['test_accuracy'].mean():.4f}")
print(f"Precision : {results['test_precision'].mean():.4f}")
print(f"Recall    : {results['test_recall'].mean():.4f}")
print(f"F1 Score  : {results['test_f1'].mean():.4f}")
print(f"ROC AUC   : {results['test_roc_auc'].mean():.4f}")

Logistic Regression Metrics:
Accuracy  : 0.8406
Precision : 0.8289
Recall    : 0.7367
F1 Score  : 0.7798
ROC AUC   : 0.8729


## Randon forest, XGBoost, CatBoost, LightGBM models

In [25]:
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'),
    'CatBoost': CatBoostClassifier(n_estimators=100, random_state=42, verbose=0),
    'LightGBM': LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)
}

for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    results = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring
    )
    
    print(f"\n{name} Metrics:")
    print(f"  Accuracy  : {results['test_accuracy'].mean():.4f}")
    print(f"  Precision : {results['test_precision'].mean():.4f}")
    print(f"  Recall    : {results['test_recall'].mean():.4f}")
    print(f"  F1 Score  : {results['test_f1'].mean():.4f}")
    print(f"  ROC AUC   : {results['test_roc_auc'].mean():.4f}")


Random Forest Metrics:
  Accuracy  : 0.8361
  Precision : 0.8086
  Recall    : 0.7514
  F1 Score  : 0.7788
  ROC AUC   : 0.8785

XGBoost Metrics:
  Accuracy  : 0.8182
  Precision : 0.7734
  Recall    : 0.7455
  F1 Score  : 0.7589
  ROC AUC   : 0.8731

CatBoost Metrics:
  Accuracy  : 0.8339
  Precision : 0.8235
  Recall    : 0.7222
  F1 Score  : 0.7691
  ROC AUC   : 0.8777


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut


LightGBM Metrics:
  Accuracy  : 0.8204
  Precision : 0.7825
  Recall    : 0.7398
  F1 Score  : 0.7597
  ROC AUC   : 0.8756


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Small learning rate → needs more trees.

In [41]:
xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", XGBClassifier(
            n_estimators=2000,
            max_depth=3,
            learning_rate=0.01,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42
        ))
    ]
)

In [42]:
param_grid = {
    "model__n_estimators": [200, 500, 1000],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.01, 0.05, 0.1]
}

random_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_grid,
    n_iter=10,
    cv=cv,
    scoring="accuracy",
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X, y)

print("Best Parameters:")
print(random_search.best_params_)

print("\nBest Score:")
print(random_search.best_score_)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best Parameters:
{'model__n_estimators': 200, 'model__max_depth': 3, 'model__learning_rate': 0.05}

Best Score:
0.8383780051471973


In [43]:
estimators = [
    ('rf', RandomForestClassifier(n_estimators=300)),
    ('xgb', XGBClassifier()),
    ('lgbm', LGBMClassifier()),
    ('cat', CatBoostClassifier(verbose=0))
]

In [44]:
stacking_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=5
)

In [45]:
stacking_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', stacking_model)
])

In [46]:
stack_scores = cross_val_score(
    stacking_pipeline,
    X,
    y,
    cv=cv,
    scoring='accuracy'
)

print("Stacking Accuracy:")
print(stack_scores.mean())

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

Stacking Accuracy:
0.8372418555018518


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [47]:
stacking_pipeline.fit(X, y)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Pclass', 'Age', 'SibSp',
                                                   'Parch', 'Fare',
                                                   'FamilySize', 'IsAlone',
                                                   'TicketGroup',
                                                   'FarePerPerson', 'Age_Class',
                                                   'Fare_Class',
                                                   'Family_Fare']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Sex', 'Embarked', 'Title'...
                                                               max_bin=None,
                                                               max_cat_threshold=None,
                                                               max_cat_to_onehot=None,
                                                               max_delta_step=None,
                                                               max_depth=None,
                                                               max_leaves=None,
                                                               min_child_weight=None,
                                                               missing=nan,
                                                               monotone_constraints=None,
                                                               multi_strategy=None,
                                                               n_estimators=None,
                                                               n_jobs=None,
                                                               num_parallel_tree=None, ...)),
                                                ('lgbm', LGBMClassifier()),
                                                ('cat',
                                                 CatBoostClassifier(verbose=0))],
                                    final_estimator=LogisticRegression()))])

In [35]:
import os
import joblib

os.makedirs("models", exist_ok=True)

joblib.dump(
    stacking_pipeline,
    'models/stacking_model.pkl'
)

print("Model saved successfully!")

Model saved successfully!


In [38]:
import os

os.makedirs("submissions", exist_ok=True)

submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': predictions.astype(int)
})

submission.to_csv(
    'submissions/stacking_submission.csv',
    index=False
)

print("Submission file saved successfully!")

Submission file saved successfully!
